In [1]:
import pandas as pd
import numpy as np
import anndata as ad
import pickle

In [2]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

## Survival

In [3]:
follow_up_df = pd.read_csv("../../data/TARGET_AML/clinical/follow_up.tsv", sep='\t')

In [4]:
follow_up_df.head()

,project.project_id,cases.case_id,cases.submitter_id,follow_ups.adverse_event,follow_ups.adverse_event_grade,follow_ups.aids_risk_factors,follow_ups.barretts_esophagus_goblet_cells_present,follow_ups.bmi,follow_ups.body_surface_area,follow_ups.cause_of_response,...,other_clinical_attributes.undescended_testis_corrected_age,other_clinical_attributes.undescended_testis_corrected_age_range,other_clinical_attributes.undescended_testis_corrected_laterality,other_clinical_attributes.undescended_testis_corrected_method,other_clinical_attributes.undescended_testis_history,other_clinical_attributes.undescended_testis_history_laterality,other_clinical_attributes.viral_hepatitis_serologies,other_clinical_attributes.viral_hepatitis_serology_tests,other_clinical_attributes.weeks_gestation_at_birth,other_clinical_attributes.weight
0,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,'--,'--,'--,'--,'--,'--,'--,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
1,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,'--,'--,'--,'--,'--,'--,'--,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
2,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,'--,'--,'--,'--,'--,'--,'--,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
3,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,'--,'--,'--,'--,'--,'--,'--,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
4,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,'--,'--,'--,'--,'--,'--,'--,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--


In [5]:
n_unique_patients = len(set(follow_up_df["cases.submitter_id"].values))
print(f"Number of unique patients: {n_unique_patients}")

Number of unique patients: 2151


In [6]:
# Clean and aggregate to one row per patient

# Replace '--' with NaN
follow_up_df.replace("'--", np.nan, inplace=True)

# Convert relevant numeric columns
numeric_cols = [
    'follow_ups.days_to_follow_up',
    'follow_ups.days_to_progression',
    'follow_ups.days_to_recurrence',
    'follow_ups.days_to_progression_free',
    'follow_ups.days_to_first_event',
]
for c in numeric_cols:
    if c in follow_up_df.columns:
        follow_up_df[c] = pd.to_numeric(follow_up_df[c], errors='coerce')

# Aggregate: take the max follow-up time and first non-null event info per patient
agg = follow_up_df.groupby('cases.submitter_id').agg(
    max_follow_up=('follow_ups.days_to_follow_up', 'max'),
    days_to_progression=('follow_ups.days_to_progression', 'min'),
    days_to_recurrence=('follow_ups.days_to_recurrence', 'min'),
    days_to_progression_free=('follow_ups.days_to_progression_free', 'max'),
    days_to_first_event=('follow_ups.days_to_first_event', 'min'),  # add this
    progression_or_recurrence=('follow_ups.progression_or_recurrence', lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan),
    first_event=('follow_ups.first_event', lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan),
).reset_index()

/tmp/ipykernel_368269/4079578624.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  follow_up_df.replace("'--", np.nan, inplace=True)


In [7]:
def compute_OS_TARGET(df):
    df = df.copy()
    dead = df['first_event'].isin(['Death', 'Death without Remission'])
    df['OS_event'] = dead.astype(float)
    df['OS_time'] = np.where(dead, df['days_to_first_event'], df['max_follow_up'])
    df['OS_time'] = pd.to_numeric(df['OS_time'], errors='coerce')
    df['OS_time'] = np.where(df['OS_time'] < 0, np.nan, df['OS_time'])
    return df

def compute_PFS_TARGET(df):
    df = df.copy()
    prog_event = df['first_event'].isin(['Relapse', 'Induction Failure', 'Death', 'Death without Remission', 'Second Malignant Neoplasm'])
    df['PFS_event'] = prog_event.astype(int)
    df['PFS_time'] = np.where(
        df['PFS_event'] == 1,
        df['days_to_first_event'],
        df['days_to_progression_free'].combine_first(df['max_follow_up'])
    )
    df['PFS_time'] = pd.to_numeric(df['PFS_time'], errors='coerce')
    df['PFS_time'] = np.where(df['PFS_time'] < 0, np.nan, df['PFS_time'])
    return df

agg = compute_OS_TARGET(agg)
agg = compute_PFS_TARGET(agg)
print(agg[['cases.submitter_id','OS_time','OS_event','PFS_time','PFS_event']].describe())

           OS_time     OS_event     PFS_time    PFS_event
count  2151.000000  2151.000000  2151.000000  2151.000000
mean   1354.071130     0.073454  1036.308229     0.516969
std     942.753146     0.260941   970.758788     0.499828
min       1.000000     0.000000     1.000000     0.000000
25%     454.500000     0.000000   216.000000     0.000000
50%    1269.000000     0.000000   596.000000     1.000000
75%    2093.000000     0.000000  1882.500000     1.000000
max    4134.000000     1.000000  4108.000000     1.000000


In [8]:
agg.head()

,cases.submitter_id,max_follow_up,days_to_progression,days_to_recurrence,days_to_progression_free,days_to_first_event,progression_or_recurrence,first_event,OS_event,OS_time,PFS_event,PFS_time
0,TARGET-20-PABGKN,95.0,NaN,NaN,NaN,95.0,NaN,Death without Remission,1.0,95.0,1,95.0
1,TARGET-20-PABHET,3632.0,NaN,NaN,NaN,3632.0,NaN,Censored,0.0,3632.0,0,3632.0
2,TARGET-20-PABHKY,2472.0,NaN,NaN,NaN,2472.0,NaN,Censored,0.0,2472.0,0,2472.0
3,TARGET-20-PABLDZ,721.0,NaN,NaN,NaN,714.0,NaN,Relapse,0.0,721.0,1,714.0
4,TARGET-20-PACDZR,455.0,NaN,NaN,NaN,377.0,NaN,Relapse,0.0,455.0,1,377.0


In [9]:
# Save aggregated follow-up data
agg.to_csv("../../data/TARGET_AML/clinical/aggregated_follow_up.csv", index=False)

## RNA

Fixes relative to the original version of this notebook:
1. `TARGET-AML.star_tpm.tsv` (Xena) stores **log2(TPM + 1)**, not TPM. Values are converted back to TPM (`2**x - 1`) before summing duplicate genes, filtering and `log1p`. Previously every step ran on the log scale and `log1p` was applied a second time.
2. Only **primary diagnosis** samples (GDC sample type `Primary ...`, barcode codes 03/09) are kept. Previously relapse, post-treatment and normal samples were included. Samples are selected *before* the expression filter and variance computation.
3. Gene symbols and protein-coding status come from **GENCODE v36** (the annotation the TPM values were quantified with; same convention as TCGA/CPTAC), replacing a live mygene query whose answer changes over time.

In [10]:
rna_path = "../../data/TARGET_AML/RNA/TARGET-AML.star_tpm.tsv"
MIN_MEAN_TPM = 0.1

In [11]:
rna_df = pd.read_csv(rna_path, sep='\t', index_col=0)
print(rna_df.shape)
print(rna_df.iloc[:3, :3])

(60660, 3063)
                    TARGET-20-PASNIY-09A  TARGET-20-PASNKC-09A  \
Ensembl_ID                                                       
ENSG00000000003.15              0.215244              0.406863   
ENSG00000000005.6               0.000000              0.000000   
ENSG00000000419.13              6.263064              6.039077   

                    TARGET-20-PASNLR-03A  
Ensembl_ID                                
ENSG00000000003.15              0.515006  
ENSG00000000005.6               0.000000  
ENSG00000000419.13              5.598210  


In [12]:
# The file holds log2(TPM + 1): undo it and check each sample sums to 1e6 TPM
tpm = np.power(2.0, rna_df) - 1.0
col_sums = tpm.sum(axis=0)
assert np.allclose(col_sums, 1e6, rtol=1e-3), 'Input is not log2(TPM+1)'
print(f'Per-sample TPM sums: {col_sums.min():,.0f} - {col_sums.max():,.0f}')

Per-sample TPM sums: 1,000,000 - 1,000,000


In [13]:
# Map versioned Ensembl IDs to GENCODE v36 gene names; sum duplicate symbols on the TPM scale
gencode = pd.read_csv('../../data/GENCODE/gencode_v36_genes.csv').set_index('gene_id')
assert tpm.index.isin(gencode.index).all()
tpm['symbol'] = gencode.loc[tpm.index, 'gene_name'].values
tpm = tpm.groupby('symbol').sum()
gene_type = gencode.drop_duplicates('gene_name').set_index('gene_name')['gene_type']
print(f'Unique gene symbols: {tpm.shape[0]}')

Unique gene symbols: 59427


### Keep primary diagnosis samples of patients with follow-up

In [14]:
pheno = pd.read_csv('../../data/TARGET_AML/clinical/TARGET-AML.clinical.tsv', sep='\t')
stype = pheno.drop_duplicates('sample').set_index('sample')['sample_type.samples']
clinical_ids = set(follow_up_df['cases.submitter_id'].dropna())

samples = tpm.columns.tolist()
sample_to_patient = {s: s.rsplit('-', 1)[0] for s in samples}  # TARGET-20-PASNIY-09A -> TARGET-20-PASNIY
print(stype.reindex(samples).fillna('missing').value_counts())

sample_type.samples
Primary Blood Derived Cancer - Bone Marrow                 1786
Bone Marrow Normal                                          394
Recurrent Blood Derived Cancer - Bone Marrow                387
Primary Blood Derived Cancer - Peripheral Blood             342
Next Generation Cancer Model                                 50
Blood Derived Normal                                         31
Recurrent Blood Derived Cancer - Peripheral Blood            29
Blood Derived Cancer - Bone Marrow, Post-treatment           25
Cell Lines                                                   18
Blood Derived Cancer - Peripheral Blood, Post-treatment       1
Name: count, dtype: int64


In [15]:
samples_to_keep = [
    s for s in samples
    if str(stype.get(s, '')).startswith('Primary') and sample_to_patient[s] in clinical_ids
]
assert all(s.split('-')[-1][:2] in {'03', '09'} for s in samples_to_keep)
n_patients = len({sample_to_patient[s] for s in samples_to_keep})
print(f'Kept {len(samples_to_keep)} primary diagnosis samples from {n_patients} patients '
      f'(of {len(samples)} samples)')
samples_df = pd.DataFrame({'sample_id': samples_to_keep,
                           'patient_id': [sample_to_patient[s] for s in samples_to_keep]})
print(samples_df['patient_id'].value_counts().value_counts().sort_index())

Kept 1953 primary diagnosis samples from 1937 patients (of 3063 samples)
count
1    1921
2      16
Name: count, dtype: int64


### Expression filter, log1p and variance on the kept cohort

In [16]:
X = tpm[samples_to_keep].T.astype(np.float32)  # samples x genes
X = X.loc[:, X.mean(axis=0) >= MIN_MEAN_TPM]
X_log = np.log1p(X)
print(f'Genes with mean TPM >= {MIN_MEAN_TPM}: {X.shape[1]}')

obs = pd.DataFrame({'patient_id': [sample_to_patient[s] for s in samples_to_keep]}, index=samples_to_keep)
var = pd.DataFrame(index=X.columns)
var['gene_symbol'] = X.columns
var['gene_type'] = gene_type.reindex(X.columns).values
var['log_variance'] = X_log.var(axis=0, ddof=0).values

adata = ad.AnnData(X=X.values, obs=obs, var=var)
log_adata = ad.AnnData(X=X_log.values.astype(np.float32), obs=obs, var=var)
print(adata)

Genes with mean TPM >= 0.1: 39715


AnnData object with n_obs × n_vars = 1953 × 39715
    obs: 'patient_id'
    var: 'gene_symbol', 'gene_type', 'log_variance'


In [17]:
adata.write_h5ad('../../data/TARGET_AML/RNA/TARGET_AML_bulk_RNA_processed.h5ad')
log_adata.write_h5ad('../../data/TARGET_AML/RNA/TARGET_AML_bulk_RNA_processed_log1p.h5ad')

### Protein-coding genes ordered by log1p variance (baseline input)

In [18]:
pc = var[var['gene_type'] == 'protein_coding']
selected_genes_TARGET = dict(pc['log_variance'].sort_values(ascending=False).items())
print(f'Selected {len(selected_genes_TARGET)} protein-coding genes')
pickle_data(selected_genes_TARGET, '../../data/TARGET_AML/TARGET_AML_genes_ordered_by_variance.pkl')

Selected 16714 protein-coding genes


In [19]:
print(f'log1p max {X_log.values.max():.2f}; protein-coding TPM sum median '
      f'{X[pc.index].sum(axis=1).median():,.0f}')

log1p max 13.22; protein-coding TPM sum median 685,715


## Clinical

In [20]:
clinical_df = pd.read_csv("../../data/TARGET_AML/clinical/clinical.tsv", sep='\t')

In [21]:
clinical_df["treatments.submitter_id"].value_counts()

treatments.submitter_id
'--                            341
TARGET-20-PANTIV_treatment       1
TARGET-20-PANTIV_treatment2      1
TARGET-20-PANTIV_treatment5      1
TARGET-20-PANTIV_treatment4      1
                              ... 
TARGET-20-PATDMY_treatment2      1
TARGET-20-PATDMY_treatment5      1
TARGET-20-PATDMY_treatment4      1
TARGET-20-PATDMY_treatment3      1
TARGET-20-PATDMY_treatment       1
Name: count, Length: 9478, dtype: int64

In [22]:
for col in clinical_df.columns:
    print(f"{col}: {clinical_df[col].nunique()} unique values")

project.project_id: 1 unique values
cases.case_id: 2492 unique values
cases.consent_type: 1 unique values
cases.days_to_consent: 1 unique values
cases.days_to_lost_to_followup: 1 unique values
cases.disease_type: 2 unique values
cases.index_date: 2 unique values
cases.lost_to_followup: 1 unique values
cases.primary_site: 3 unique values
cases.submitter_id: 2492 unique values
demographic.age_at_index: 31 unique values
demographic.age_is_obfuscated: 2 unique values
demographic.cause_of_death: 1 unique values
demographic.cause_of_death_source: 1 unique values
demographic.country_of_birth: 1 unique values
demographic.country_of_residence_at_enrollment: 1 unique values
demographic.days_to_birth: 1802 unique values
demographic.days_to_death: 518 unique values
demographic.demographic_id: 2190 unique values
demographic.education_level: 1 unique values
demographic.ethnicity: 5 unique values
demographic.gender: 5 unique values
demographic.marital_status: 1 unique values
demographic.occupation_du

diagnoses.sites_of_involvement_count: 1 unique values
diagnoses.submitter_id: 2190 unique values
diagnoses.supratentorial_localization: 1 unique values
diagnoses.synchronous_malignancy: 1 unique values
diagnoses.tissue_or_organ_of_origin: 2 unique values
diagnoses.tumor_burden: 1 unique values
diagnoses.tumor_confined_to_organ_of_origin: 1 unique values
diagnoses.tumor_depth: 1 unique values
diagnoses.tumor_focality: 1 unique values
diagnoses.tumor_grade: 2 unique values
diagnoses.tumor_grade_category: 1 unique values
diagnoses.tumor_of_origin: 1 unique values
diagnoses.tumor_regression_grade: 1 unique values
diagnoses.uicc_clinical_m: 1 unique values
diagnoses.uicc_clinical_n: 1 unique values
diagnoses.uicc_clinical_stage: 1 unique values
diagnoses.uicc_clinical_t: 1 unique values
diagnoses.uicc_pathologic_m: 1 unique values
diagnoses.uicc_pathologic_n: 1 unique values
diagnoses.uicc_pathologic_stage: 1 unique values
diagnoses.uicc_pathologic_t: 1 unique values
diagnoses.uicc_staging_

In [23]:
# Get gender value counts
clinical_df["cases.primary_site"].value_counts()

cases.primary_site
Hematopoietic and reticuloendothelial systems    9748
Unknown                                            69
'--                                                 1
Name: count, dtype: int64

In [24]:
clinical_df.head()

,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TARGET-AML,002e6505-2e1f-4606-ba1e-0201bf9e53a6,'--,'--,'--,Myeloid Leukemias,'--,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PARPSX,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
1,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,1e05395d-e7df-4850-aaaa-9fb9fa660db2,'--,yes,'--,'--,'--
2,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,56209277-05d1-4833-b0d2-b3bd1276be4c,'--,no,'--,'--,"Stem Cell Transplantation, NOS"
3,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,5d50ad36-2490-474b-b334-c928f93355e8,'--,yes,'--,'--,"Pharmaceutical Therapy, NOS"
4,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,be653481-3d5c-4cad-8408-c10b1b61e017,'--,yes,'--,'--,'--
